# Torch Available

In [1]:
import torch

torch.cuda.is_available()

False

# Test Cartpole Overfit

In [2]:
%load_ext autoreload
%autoreload 2

In [2]:
import gym
import procgen

In [3]:
from shimmy.openai_gym_compatibility import GymV26CompatibilityV0
cart_env = GymV26CompatibilityV0("CartPole-v1", make_kwargs={"render_mode": "rgb_array"})

In [4]:
cart_env.reset()

(array([-0.00895673, -0.02353574,  0.02960008,  0.00293045], dtype=float32),
 {})

In [8]:
from stable_baselines3.common import env_checker
env_checker.check_env(cart_env)

In [9]:
observation, info = cart_env.reset(seed=42)
total_reward = 0
for _ in range(1000):
    action = cart_env.action_space.sample()  # this is where you would insert your policy
    observation, reward, terminated, truncated, info = cart_env.step(action)
    total_reward += reward
    if terminated or truncated:
        observation, info = cart_env.reset()
cart_env.close()
print(total_reward)

1000.0


In [11]:
from stable_baselines3 import PPO
from impala import ImpalaActorCriticPolicy

model = PPO("MlpPolicy", cart_env, verbose=1, device="cuda")
model.learn(5_000_000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 21.9     |
|    ep_rew_mean     | 21.9     |
| time/              |          |
|    fps             | 1018     |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 29.2        |
|    ep_rew_mean          | 29.2        |
| time/                   |             |
|    fps                  | 911         |
|    iterations           | 2           |
|    time_elapsed         | 4           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.008950675 |
|    clip_fraction        | 0.105       |
|    clip_range           | 0.2         |
|    entropy_loss   

KeyboardInterrupt: 

# PRocgen

In [3]:
from shimmy.openai_gym_compatibility import GymV21CompatibilityV0
# cart_env = GymV26CompatibilityV0("procgen:procgen-coinrun-v0", make_kwargs={"render_mode": "rgb_array"})
coin_non_env = GymV21CompatibilityV0("procgen:procgen-coinrun-v0", make_kwargs={"num_levels": 1, "start_level": 0, "distribution_mode": "easy"})


In [6]:
import gymnasium as gym
from procgen import ProcgenEnv
from shimmy.openai_gym_compatibility import GymV21CompatibilityV0
from stable_baselines3.common import env_checker

# Custom wrapper to ensure compatibility with Stable Baselines3 and the 'seed' parameter
class CustomProcgenEnvWrapper(gym.Env):
    def __init__(self, env):
        self.env = env

    def reset(self, *args, **kwargs):
        # Ignore the seed parameter and just call the original reset method
        if 'seed' in kwargs:
            del kwargs['seed']  # Ignore the seed parameter, but still accept it
        return self.env.reset(*args, **kwargs)

    def step(self, action):
        state, reward, terminated, truncated, info =  self.env.step(action)
        reward = float(reward)
        terminated = bool(terminated)
        truncated = bool(truncated)
        return (state, reward, terminated, truncated, info)

    def render(self):
        return self.env.render()

    @property
    def observation_space(self):
        return self.env.observation_space
    
    @property
    def action_space(self):
        return self.env.action_space

# Create the original Procgen environment
coin_non_env = GymV21CompatibilityV0("procgen:procgen-coinrun-v0", make_kwargs={
    "num_levels": 1,
    "start_level": 0,
    "distribution_mode": "easy"
})

# Apply the custom wrapper
wrapped_env = CustomProcgenEnvWrapper(coin_non_env)

# # Optionally, vectorize the environment
# from stable_baselines3.common.vec_env import DummyVecEnv
# vec_env = DummyVecEnv([lambda: wrapped_env])

env_checker.check_env(wrapped_env)
# Now you can train your model, with `reset()` accepting `seed` but ignoring it


/storage/home/hcoda1/4/rmehta98/.conda/envs/rl_env/lib/python3.9/site-packages/gym/utils/passive_env_checker.py:174: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed a `seed` instead of using `Env.seed` for resetting the environment random number generator.
  logger.warn(
/storage/home/hcoda1/4/rmehta98/.conda/envs/rl_env/lib/python3.9/site-packages/gym/utils/passive_env_checker.py:187: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed `options` to allow the environment initialisation to be passed additional information.
  logger.warn(
/storage/home/hcoda1/4/rmehta98/.conda/envs/rl_env/lib/python3.9/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(
/storage/home/hcoda1/4/rmehta9

In [14]:
from stable_baselines3 import PPO
from impala import ImpalaActorCriticPolicy

model = PPO(ImpalaActorCriticPolicy, wrapped_env, verbose=1)
model.learn(5_000_000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.
>> run initialize
{'pi': [256], 'vf': [256]}
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 330      |
|    ep_rew_mean     | 0        |
| time/              |          |
|    fps             | 648      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 394         |
|    ep_rew_mean          | 0           |
| time/                   |             |
|    fps                  | 262         |
|    iterations           | 2           |
|    time_elapsed         | 15          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.007107543 |
|    clip_fraction

KeyboardInterrupt: 

In [47]:
# cannot use the sym link to save
model.save("/storage/coda1/p-adelarue3/0/rmehta98/impala_sanity")

In [48]:
obs, _ = wrapped_env.reset()
model.predict(obs)

(array(8), None)

# Test Model

In [52]:
from stable_baselines3 import PPO
from impala import ImpalaActorCriticPolicy

model_loaded = PPO.load("/storage/coda1/p-adelarue3/0/rmehta98/impala_sanity", env=wrapped_env, verbose=True, policy=ImpalaActorCriticPolicy)

# model_loaded = PPO(ImpalaActorCriticPolicy, wrapped_env, verbose=1)
# model_loaded.load()

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.
>> run initialize
{'pi': [256], 'vf': [256]}


In [53]:

for ep in range(20): # number of episodes
    terminated, truncated = False, False
    total_reward = 0
    obs, _ = wrapped_env.reset()
    step = 0
    while not terminated and not truncated:
        action, _ = model_loaded.predict(obs)
        # out = wrapped_env.step(action)
        obs, reward, truncated, terminated, _ = wrapped_env.step(action)
        total_reward += reward
        step += 1
    
    print(f"Episode {ep + 1} terminated with reward of {total_reward} (step {step})")

Episode 1 terminated with reward of 10.0 (step 75)
Episode 2 terminated with reward of 10.0 (step 77)
Episode 3 terminated with reward of 10.0 (step 76)
Episode 4 terminated with reward of 10.0 (step 75)
Episode 5 terminated with reward of 10.0 (step 77)
Episode 6 terminated with reward of 10.0 (step 75)
Episode 7 terminated with reward of 10.0 (step 76)
Episode 8 terminated with reward of 10.0 (step 75)
Episode 9 terminated with reward of 0.0 (step 79)
Episode 10 terminated with reward of 10.0 (step 76)
Episode 11 terminated with reward of 10.0 (step 75)
Episode 12 terminated with reward of 10.0 (step 75)
Episode 13 terminated with reward of 10.0 (step 76)
Episode 14 terminated with reward of 10.0 (step 76)
Episode 15 terminated with reward of 10.0 (step 77)
Episode 16 terminated with reward of 10.0 (step 75)
Episode 17 terminated with reward of 10.0 (step 75)
Episode 18 terminated with reward of 10.0 (step 77)
Episode 19 terminated with reward of 10.0 (step 75)
Episode 20 terminated 